In [1]:
%cd /Users/soniaatre/Cornell2025/Research/PublicView/sycophancy-stylized-facts
%pwd

/Users/soniaatre/Cornell2025/Research/PublicView/sycophancy-stylized-facts


'/Users/soniaatre/Cornell2025/Research/PublicView/sycophancy-stylized-facts'

In [31]:
import numpy as np
import scipy.sparse as sp
import json

def load_npz(file_path):
    """Load npz sparse activation file into a dictionary."""
    data = np.load(file_path)
    N, F = int(data['shape'][0]), int(data['shape'][1])
    return sp.csr_matrix((data['values'].astype(np.float32),
                       (data['row_indices'], data['feature_indices'])),
                        shape=(N, F)).tocsc()


def load_jsonl(file_path):
    """Load jsonl file into a list of dictionaries."""
    with open(file_path, 'r') as file:
        return [json.loads(line) for line in file]


def clean_data(data, min_char=40):
    """Remove automated bot, deleted/removed, and comments less than min_char characters."""
    bad_authors = {'AutoModerator', 'ChatGPT-ModTeam'}
    seen = set()
    m = np.zeros(len(data), dtype=bool)
    for i, row in enumerate(data):
        body = row.get('body', '').strip()
        ok = (row.get('author') not in bad_authors
              and len(body) >= min_char
              and body not in ('[removed]', '[deleted]', '')
              and body not in seen)
        m[i] = ok
        if ok:
            seen.add(body)
    return m


def neuron_sampling(matrix, data, neuron_idx):
    activations = matrix[:, neuron_idx].toarray().ravel()
    clean = clean_data(data)
    active = np.flatnonzero((activations > 0) & clean)
    inactive = np.flatnonzero((activations == 0) & clean)
    active = active[np.argsort(-activations[active])]
    return {"act": activations, "active": active, "inactive": inactive}


import re

def build_pools(matrix, data, neuron_idx, n_neg=120, seed=42):
    rng = np.random.default_rng(seed)
    activations = matrix[:, neuron_idx].toarray().ravel()
    clean = clean_data(data)
    active = np.flatnonzero(activations > 0 & clean)
    inactive = np.flatnonzero(activations == 0 & clean)
    active = active[np.argsort(-activations[active])]

    top_dec = active[:max(1, len(active)//10)]
    # generate negatives that match this keyword (make more difficult to distinguish from positives)
    kw = re.compile(r'sycophan', re.I) 
    inactive_kw = np.array([i for i in inactive if kw.search(data[i]['body'])])
    n_kw = min(n_neg//2, len(inactive_kw))
    neg_kw = rng.choice(inactive_kw, n_kw, replace=False) if n_kw else np.array([])
    rest = np.setdiff1d(inactive, neg_kw)
    neg_rand = rng.choice(rest, min(n_neg - n_kw, len(rest)), replace=False)

    def rows(idx, with_rank):
        out = []
        for r, i in enumerate(idx):
            d = {'doc_id': data[i]['id'], 'text': data[i]['body'],
                 'created_utc': data[i].get('created_utc'), 'activation': float(activations[i])}
            if with_rank: d['rank'] = r + 1
            out.append(d)
        return out
    
    pools = {
        'pos_dec': rows(top_dec, True),
        'neg_kw': rows(neg_kw, False),
        'neg_rand': rows(neg_rand, False),
    }
    json.dump(pools, open(f'data/pools_{neuron_idx}.json', 'w'), indent=4)
    print({k: len(v) for k, v in pools.items()})
    return pools

In [38]:
neuron_idx = 72476
npz_file = "data/comments_relating_sycophan_gemini_d131072_sparse_activations.npz"
data_file = "data/r_chatGPT_comments_relating_sycophan_2022-12-22_2026-8-7.jsonl"

matrix = load_npz(npz_file)
data = load_jsonl(data_file)

pools_39744 = build_pools(matrix, data, neuron_idx)


def safe_sample(rng, pool, n):
    """Sample without replacement from pool - retrieves n samples or full pool if n > pool size"""
    return rng.choice(pool, size=min(n, len(pool)), replace=False)

# randomly sample 30 positive from pos_dec and 30 negative from neg_kw + neg_rand
rng = np.random.default_rng(42)
pos_dec_sample = safe_sample(rng, pools_39744['pos_dec'], 30)
neg_combined = np.concatenate([pools_39744['neg_kw'], pools_39744['neg_rand']])
neg_combined_sample = safe_sample(rng, neg_combined, 30)

{'pos_dec': 313, 'neg_kw': 60, 'neg_rand': 60}


In [39]:
pos_dec_sample = json.dumps(pos_dec_sample.tolist(), indent=4)
neg_combined_sample = json.dumps(neg_combined_sample.tolist(), indent=4)
print(pos_dec_sample)
print("-"*100)
print(neg_combined_sample)

[
    {
        "doc_id": "mps8tpn",
        "text": "Yes this is a documented phenomena. In this [Verge article from one day ago](https://www.theverge.com/tech/657409/chat-gpt-sycophantic-responses-gpt-4o-sam-altman), Sam Altman commented on it saying stating model needs to be [tweaked to fix this.](https://www.windowscentral.com/software-apps/openai-sam-altman-admits-chatgpt-glazes-too-much)\n\nThis seems pretty dangerous for people with mental illness and delusions of grandeur. I\u2019m not going to use it again until they fix it. This kind emotional manipulation is weird.",
        "created_utc": 1745985330,
        "activation": 0.09576416015625,
        "rank": 199
    },
    {
        "doc_id": "mprvx55",
        "text": "A recent update to GPT-4o made it very eager to please. It constantly glazed user ideas, calling the user genius. Even Sam Altman admitted it was sycophantic behavior. \n\nA few days ago, they released a partial fix, by updating the system prompt to tell it not